# LeetCode #1094: Car Pooling

https://leetcode.com/problems/car-pooling/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \cdot \text{maxStop})$ | $O(\text{maxStop})$ |
| **Optimal: Difference Array ★** | $O(n + \text{maxStop})$ | $O(\text{maxStop})$ |

---

## Understanding the Methods

### Brute Force
For each trip, increment a passengers-at-stop counter for every stop in the trip's range. Then scan all stops to check if any exceed capacity. $O(n \cdot \text{maxStop})$ in the worst case.

### Optimal: Difference Array ★
Use a difference array of size 1001 (stops are ≤ 1000). For each trip [numPassengers, from, to], add numPassengers at index `from` and subtract at index `to` (passengers exit before the stop, not after). A prefix sum scan over the array reconstructs actual passenger counts at every stop. If any prefix sum exceeds capacity, return false. Total work is $O(n + \text{maxStop})$ — one pass per trip to update the difference array, one pass to check.

**Constraints:**
* 1 <= trips.length <= 1000
* trips[i].length == 3
* 1 <= numPassengers_i <= 100
* 0 <= from_i < to_i <= 1000
* 1 <= capacity <= 10^5

## Solutions
### C#

In [ ]:
// Difference array: mark passenger changes at trip endpoints, then prefix-sum check
public class Solution {
    public bool CarPooling(int[][] trips, int capacity) {
        // Difference array indexed by stop number (stops go up to 1000)
        int[] diff = new int[1001];
        foreach (var trip in trips) {
            int num = trip[0], from = trip[1], to = trip[2];
            // Passengers board at 'from' and exit before 'to'
            diff[from] += num;
            diff[to] -= num;
        }

        // Running prefix sum = actual passenger count at each stop
        int current = 0;
        foreach (int delta in diff) {
            current += delta;
            if (current > capacity) return false;
        }
        return true;
    }
}

### Python

In [ ]:
# Difference array: mark passenger changes at trip endpoints, then prefix-sum check
from typing import List

class Solution:
    def carPooling(self, trips: List[List[int]], capacity: int) -> bool:
        # Difference array sized for all possible stops (0–1000)
        diff = [0] * 1001
        for num, frm, to in trips:
            # Passengers add to load at 'from', remove before reaching 'to'
            diff[frm] += num
            diff[to] -= num

        # Prefix sum converts deltas back to actual occupancy at each stop
        current = 0
        for delta in diff:
            current += delta
            if current > capacity:
                return False
        return True

### Go

In [ ]:
// Difference array: mark passenger changes at trip endpoints, then prefix-sum check
package main

func carPooling(trips [][]int, capacity int) bool {
    // Difference array — stops are in range [0, 1000]
    diff := [1001]int{}
    for _, trip := range trips {
        num, from, to := trip[0], trip[1], trip[2]
        // Board at 'from', alight just before 'to'
        diff[from] += num
        diff[to] -= num
    }

    // Accumulate prefix sum to get passenger count at each stop
    current := 0
    for _, delta := range diff {
        current += delta
        if current > capacity {
            return false
        }
    }
    return true
}

### Rust

In [ ]:
// Difference array: mark passenger changes at trip endpoints, then prefix-sum check
impl Solution {
    pub fn car_pooling(trips: Vec<Vec<i32>>, capacity: i32) -> bool {
        // Difference array over stops 0..=1000
        let mut diff = [0i32; 1001];
        for trip in &trips {
            let (num, from, to) = (trip[0], trip[1] as usize, trip[2] as usize);
            // Passengers board at 'from' and leave at 'to' (before that stop)
            diff[from] += num;
            diff[to] -= num;
        }

        // Running prefix sum = current occupancy; fail fast if capacity exceeded
        let mut current = 0i32;
        for &delta in &diff {
            current += delta;
            if current > capacity {
                return false;
            }
        }
        true
    }
}

## Example Scenarios

**1. Common Case** — Two trips, capacity sufficient

**Input:** `trips = [[2,1,5],[3,3,7]], capacity = 5`
Diff: +2 at stop 1, -2 at stop 5; +3 at stop 3, -3 at stop 7. Prefix sums: [0,2,2,5,5,3,3,0]. Max = 5 = capacity. Returns `true`.

**2. Slightly Complex** — Trips overlap, capacity exceeded

**Input:** `trips = [[2,1,5],[3,3,7]], capacity = 4`
Same difference array; prefix sum peaks at 5 at stop 3. 5 > 4 returns `false` immediately upon reaching stop 3.

**3. Edge Case: Time Factor** — 1000 trips all spanning the full route

**Input:** `trips = [[1,0,1000]] * 1000, capacity = 999`
After all updates, diff[0] = 1000, diff[1000] = -1000. Prefix sum reaches 1000 at stop 0. 1000 > 999 returns `false` after a single scan step.

**4. Edge Case: Space Factor** — Trips scattered across all 1001 stops

**Input:** 1000 trips each with different from/to values covering stops 0–1000
The diff array is always exactly 1001 entries regardless of input size — $O(1)$ space relative to stop count.

**5. Almost-Impossible but Plausible** — All passengers exit at the last stop simultaneously

**Input:** `trips = [[100, 0, 1000]] * 10, capacity = 1000`
Total passengers = 1000 = capacity. Diff: +1000 at stop 0, -1000 at stop 1000. Prefix sum stays at 1000 for stops 0–999, drops to 0 at stop 1000. Exactly at capacity — returns `true`.